In [1]:
#import libraries
import os
import numpy as np
import pandas as pd
import timeit

from sklearn.model_selection import train_test_split, RepeatedKFold, GroupKFold, GridSearchCV
from sklearn.linear_model import Ridge, RidgeCV, LassoCV, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import geopandas as gpd
import seaborn as sns
import matplotlib.cm
import matplotlib.pyplot as plt
%matplotlib inline

## import MOSAIKS features

In [2]:
# import the first file 
df1 = pd.read_pickle(r'C:\HDRO\Stats\Stats2\MOSAIKS\DHS_DHS_dense_DHSID.p')
df1['ID']= df1['DHSID'].str.slice(0,2)
df1['year']= df1['DHSID'].str.slice(2,6)

In [3]:
# import the additinal file with 8 countries 
df2 = pd.read_pickle(r'C:\HDRO\Stats\Stats2\MOSAIKS\DHS_additional_global_dense_DHSID.p')
df2.columns = df2.columns.str.replace('dhsid', 'DHSID')
df2['ID']= df2['DHSID'].str.slice(0,2)
df2['year']= df2['DHSID'].str.slice(2,6)

In [4]:
# merge two datasets - this is the final dataset
df3 = pd.concat([df1, df2])
df4 = df3.drop(columns=['continent', 'ID', 'year'])

## Import NL features

In [5]:
nl = pd.read_pickle("C:/HDRO/Stats/Stats2/MOSAIKS/all_dhs_dmsp_nightlight_features_20_bins_GPW_pop_weighted.p")

In [6]:
nl

,perc_pixels_in_bin_0,perc_pixels_in_bin_1,perc_pixels_in_bin_2,perc_pixels_in_bin_3,perc_pixels_in_bin_4,perc_pixels_in_bin_5,perc_pixels_in_bin_6,perc_pixels_in_bin_7,perc_pixels_in_bin_8,perc_pixels_in_bin_9,perc_pixels_in_bin_10,perc_pixels_in_bin_11,perc_pixels_in_bin_12,perc_pixels_in_bin_13,perc_pixels_in_bin_14,perc_pixels_in_bin_15,perc_pixels_in_bin_16,perc_pixels_in_bin_17,perc_pixels_in_bin_18,perc_pixels_in_bin_19
DHSID,,,,,,,,,,,,,,,,,,,,
CM201800000018,0.0,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
EG201401460501,0.0,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
GN201800000186,0.0,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
KE201400000266,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
PK201700000094,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
RW201900000494,0.0,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
RW201900000495,0.0,0.834223,0.038776,0.048623,0.041006,0.024467,0.012905,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
RW201900000496,0.0,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Import income

In [7]:
# import income file
df_list = []
direct2 = ("C:/HDRO/Stats/Stats2/geo-inc/")
for file in os.listdir(direct2 + 'geo-income_module'):
    d = pd.read_stata(direct2 + 'geo-income_module/' + file)
    df_list.append(d)
df_inc = pd.concat(df_list)

In [8]:
income = df_inc.groupby("DHSID")[["lognormal_inc_ind_wiid"]].agg(np.nanmean).dropna()

In [9]:
income

,lognormal_inc_ind_wiid
DHSID,
AL201700000001,18942.701172
AL201700000002,18275.437500
AL201700000003,19139.130859
AL201700000004,20030.707031
AL201700000005,22517.060547
...,...
ZM201800000541,7729.175293
ZM201800000542,894.005615
ZM201800000543,3261.351807


## Import MPI data

In [10]:
#mpi file
direct = ("C:/HDRO/Stats/Stats2/MOSAIKS/")
df_mpi = pd.read_stata(direct + "mpi_geo37.dta")

In [11]:
dep_score = df_mpi.groupby("DHSID")[["weighted_deprivation_score"]].agg(np.nanmean).dropna()

In [12]:
dep_score

,weighted_deprivation_score
DHSID,
AL201700000001,0.019204
AL201700000002,0.009615
AL201700000003,0.000000
AL201700000004,0.024024
AL201700000005,0.004274
...,...
ZM201800000541,0.178197
ZM201800000542,0.367798
ZM201800000543,0.284661


In [13]:
# 10 indicators
indicator_cols = ['d_cm', 'd_nutr', 'd_satt', 'd_educ',  'd_elct', 'd_wtr', 'd_sani', 'd_hsg', 'd_ckfl', 'd_asst']

In [14]:
indi = df_mpi.groupby("DHSID")[indicator_cols].agg(np.nanmean).dropna()

In [15]:
indi

,d_cm,d_nutr,d_satt,d_educ,d_elct,d_wtr,d_sani,d_hsg,d_ckfl,d_asst
DHSID,,,,,,,,,,
AL201700000001,0.000000,0.074074,0.000000,0.022989,0.000000,0.000000,0.000000,0.000000,0.045977,0.000000
AL201700000002,0.000000,0.000000,0.000000,0.019231,0.000000,0.038462,0.000000,0.000000,0.076923,0.000000
AL201700000003,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
AL201700000004,0.000000,0.000000,0.051282,0.038462,0.000000,0.000000,0.000000,0.000000,0.141026,0.000000
AL201700000005,0.000000,0.000000,0.000000,0.023256,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
ZM201800000541,0.150943,0.339623,0.066038,0.000000,0.000000,0.000000,0.764151,0.028302,0.726415,0.018868
ZM201800000542,0.000000,0.314815,0.203704,0.129630,1.000000,0.759259,0.722222,1.000000,1.000000,0.194444
ZM201800000543,0.000000,0.274336,0.221239,0.079646,0.769912,0.415929,0.752212,0.389381,0.946903,0.123894


## Import IWI data

In [16]:
#iwi data
iwi = pd.read_csv(r'C:\HDRO\Stats\Stats2\MOSAIKS\mean_IWI.csv')
iwi = iwi.groupby("DHSID")[["IWI"]].mean()

In [17]:
iwi

,IWI
DHSID,
AL201700000001,89.465391
AL201700000002,86.485039
AL201700000003,87.642180
AL201700000004,86.776051
AL201700000005,89.517279
...,...
ZW201500000396,38.506412
ZW201500000397,41.553372
ZW201500000398,31.797095


In [18]:
# Keep only the rows with unique DHSIDs
df_mpi_v1 = df_mpi.drop_duplicates(subset='DHSID', keep='first')

# Select only the 'dhsid' and 'rural/urban' columns from the unique DataFrame
df_mpi_v1 = df_mpi_v1[['DHSID', 'URBAN_RURA']]

# Print the resulting DataFrame
print(df_mpi_v1)

                  DHSID URBAN_RURA
0        AO201500000001          R
63       AO201500000002          U
128      AO201500000003          U
205      AO201500000004          R
246      AO201500000005          R
...                 ...        ...
5287699  PH201700000301          U
5393118  ZA201700000082          U
5396687  ZA201700000211          U
5405211  ZA201700000529          U
5407550  ZA201700000627          U

[53132 rows x 2 columns]


## join all data

In [19]:
merge1 = pd.merge(df4, nl, on=['DHSID'], how="inner")

In [20]:
merge2 = pd.merge(merge1, income, on=['DHSID'], how="inner")
merge3 = pd.merge(merge2, dep_score, on=['DHSID'], how="inner")
merge4 = pd.merge(merge3, indi, on=['DHSID'], how = "inner")

#merge4 = pd.merge(merge3, iwi, on=['DHSID'], how="inner")

In [21]:
merge4 = pd.merge(merge3, df_mpi_v1, on=['DHSID'], how = "inner")

In [22]:
#get the log of income
merge4['loginc'] = np.log(merge4['lognormal_inc_ind_wiid'])

In [47]:
#checking for missing
merge4.isnull().sum()

country                       0
URBAN_RURAL                   0
X_0                           0
X_1                           0
X_2                           0
                             ..
perc_pixels_in_bin_18         0
perc_pixels_in_bin_19         0
lognormal_inc_ind_wiid        0
weighted_deprivation_score    0
loginc                        0
Length: 4025, dtype: int64

In [24]:
#getting the ID
merge4['ID']= merge4['DHSID'].str.slice(0,2)

# #getiing the numeric id
# merge4['ID_n'] = merge4['ID'].factorize()[0] + 1

# #moving the id in the front
# merge4.insert(1, 'ID_n1', merge4['ID_n'])

In [25]:
merge4.insert(1, 'country', merge4['ID'])
merge4 = merge4.drop(['DHSID', "ID"], axis=1)
merge4

,country,X_0,X_1,X_2,X_3,X_4,X_5,X_6,X_7,X_8,...,perc_pixels_in_bin_14,perc_pixels_in_bin_15,perc_pixels_in_bin_16,perc_pixels_in_bin_17,perc_pixels_in_bin_18,perc_pixels_in_bin_19,lognormal_inc_ind_wiid,weighted_deprivation_score,URBAN_RURA,loginc
0,AL,0.247140,0.503159,0.114064,0.203914,0.353395,0.727176,0.218433,0.306100,0.130447,...,0.235019,0.082250,0.393638,0.045660,0.000000,0.000000,18942.701172,0.019204,U,9.849174
1,AL,0.218926,0.439225,0.103380,0.159651,0.315144,0.711148,0.225437,0.287943,0.098494,...,0.201556,0.110626,0.306916,0.000000,0.000000,0.000000,18275.437500,0.009615,U,9.813313
2,AL,0.239452,0.515465,0.107474,0.204323,0.362553,0.710463,0.200439,0.286575,0.131596,...,0.229501,0.081326,0.306956,0.000000,0.000000,0.000000,19139.130859,0.000000,U,9.859490
3,AL,0.230399,0.462435,0.107077,0.179146,0.332637,0.714292,0.223447,0.293186,0.113443,...,0.173535,0.095246,0.359493,0.052875,0.000000,0.000000,20030.707031,0.024024,U,9.905022
4,AL,0.247140,0.503159,0.114064,0.203914,0.353395,0.727176,0.218433,0.306100,0.130447,...,0.235019,0.082250,0.393638,0.045660,0.000000,0.000000,22517.060547,0.004274,U,10.022029
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52633,RW,0.108982,0.350792,0.034645,0.137668,0.199491,0.341322,0.128825,0.150607,0.096750,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1787.015991,0.287234,R,7.488303
52634,RW,0.138619,0.435800,0.039662,0.223957,0.251421,0.378330,0.129759,0.135954,0.172089,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1555.878784,0.369976,R,7.349796
52635,RW,0.076103,0.296412,0.011021,0.264386,0.153407,0.179054,0.067392,0.056701,0.228597,...,0.015896,0.074712,0.000000,0.247376,0.140741,0.441666,3548.524414,0.210101,U,8.174287
52636,RW,0.149145,0.475971,0.049168,0.185751,0.274041,0.468409,0.161029,0.185322,0.124372,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3598.289551,0.247942,U,8.188214


In [26]:
unique_values = merge4['country'].unique()

In [27]:
unique_values

array(['AL', 'AM', 'AO', 'BJ', 'BU', 'CM', 'EG', 'ET', 'GA', 'GN', 'GU',
       'HT', 'JO', 'KE', 'KM', 'LB', 'ML', 'MM', 'NG', 'PH', 'PK', 'SL',
       'SN', 'TJ', 'TL', 'TZ', 'UG', 'ZA', 'ZM', 'BF', 'GM', 'IA', 'KH',
       'MR', 'MZ', 'NI', 'RW'], dtype=object)

In [35]:
merge4.insert(1, 'URBAN_RURAL', merge4['URBAN_RURA'])
merge4 = merge4.drop('URBAN_RURA', axis=1)

ValueError: cannot insert URBAN_RURAL1, already exists

In [42]:
#merge4 = merge4.rename(columns={'URBAN_RURAL1': 'URBAN_RURAL'})
#merge4 = merge4.drop('URBAN_RURA', axis=1)
merge4

,country,URBAN_RURAL,X_0,X_1,X_2,X_3,X_4,X_5,X_6,X_7,...,perc_pixels_in_bin_13,perc_pixels_in_bin_14,perc_pixels_in_bin_15,perc_pixels_in_bin_16,perc_pixels_in_bin_17,perc_pixels_in_bin_18,perc_pixels_in_bin_19,lognormal_inc_ind_wiid,weighted_deprivation_score,loginc
0,AL,U,0.247140,0.503159,0.114064,0.203914,0.353395,0.727176,0.218433,0.306100,...,0.088234,0.235019,0.082250,0.393638,0.045660,0.000000,0.000000,18942.701172,0.019204,9.849174
1,AL,U,0.218926,0.439225,0.103380,0.159651,0.315144,0.711148,0.225437,0.287943,...,0.108216,0.201556,0.110626,0.306916,0.000000,0.000000,0.000000,18275.437500,0.009615,9.813313
2,AL,U,0.239452,0.515465,0.107474,0.204323,0.362553,0.710463,0.200439,0.286575,...,0.079555,0.229501,0.081326,0.306956,0.000000,0.000000,0.000000,19139.130859,0.000000,9.859490
3,AL,U,0.230399,0.462435,0.107077,0.179146,0.332637,0.714292,0.223447,0.293186,...,0.093172,0.173535,0.095246,0.359493,0.052875,0.000000,0.000000,20030.707031,0.024024,9.905022
4,AL,U,0.247140,0.503159,0.114064,0.203914,0.353395,0.727176,0.218433,0.306100,...,0.088234,0.235019,0.082250,0.393638,0.045660,0.000000,0.000000,22517.060547,0.004274,10.022029
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52633,RW,R,0.108982,0.350792,0.034645,0.137668,0.199491,0.341322,0.128825,0.150607,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1787.015991,0.287234,7.488303
52634,RW,R,0.138619,0.435800,0.039662,0.223957,0.251421,0.378330,0.129759,0.135954,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1555.878784,0.369976,7.349796
52635,RW,U,0.076103,0.296412,0.011021,0.264386,0.153407,0.179054,0.067392,0.056701,...,0.000000,0.015896,0.074712,0.000000,0.247376,0.140741,0.441666,3548.524414,0.210101,8.174287
52636,RW,U,0.149145,0.475971,0.049168,0.185751,0.274041,0.468409,0.161029,0.185322,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3598.289551,0.247942,8.188214


## within country analysis

In [43]:
# define a function to demean columns
def demean_column(grouped_col):
    return grouped_col - grouped_col.mean()

# group by the 'country' column and apply the demean function to each column
demeaned_df = merge4.iloc[:, 3:4025].groupby(merge4['country']).apply(demean_column)

#joining the countyr column back
demeaned_df = demeaned_df.join(merge4['country'])

In [44]:
demeaned_df

,X_1,X_2,X_3,X_4,X_5,X_6,X_7,X_8,X_9,X_10,...,perc_pixels_in_bin_14,perc_pixels_in_bin_15,perc_pixels_in_bin_16,perc_pixels_in_bin_17,perc_pixels_in_bin_18,perc_pixels_in_bin_19,lognormal_inc_ind_wiid,weighted_deprivation_score,loginc,country
0,0.084621,0.016802,0.034896,0.050608,0.057155,0.021160,-0.029396,0.021411,0.051269,0.051879,...,0.206662,0.057360,0.356938,-0.002086,-0.041036,-0.124213,5244.280273,-0.030550,0.437517,AL
1,0.020687,0.006118,-0.009367,0.012356,0.041126,0.028163,-0.047553,-0.010542,0.001740,0.007044,...,0.173199,0.085735,0.270216,-0.047746,-0.041036,-0.124213,4577.016602,-0.040139,0.401657,AL
2,0.096926,0.010211,0.035305,0.059765,0.040441,0.003166,-0.048921,0.022561,0.033992,0.042326,...,0.201144,0.056436,0.270256,-0.047746,-0.041036,-0.124213,5440.709961,-0.049755,0.447834,AL
3,0.043896,0.009814,0.010128,0.029849,0.044270,0.026173,-0.042310,0.004407,0.017624,0.026070,...,0.145179,0.070356,0.322793,0.005129,-0.041036,-0.124213,6332.286133,-0.025731,0.493365,AL
4,0.084621,0.016802,0.034896,0.050608,0.057155,0.021160,-0.029396,0.021411,0.051269,0.051879,...,0.206662,0.057360,0.356938,-0.002086,-0.041036,-0.124213,8818.639648,-0.045481,0.610373,AL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52633,-0.040447,0.001694,-0.118003,-0.024413,0.025389,0.023106,0.032358,-0.109825,-0.013265,-0.081719,...,-0.005089,-0.007437,-0.004712,-0.009241,-0.009473,-0.038477,-399.525513,-0.025489,-0.032187,RW
52634,0.044561,0.006710,-0.031714,0.027516,0.062397,0.024040,0.017706,-0.034486,0.037715,0.004387,...,-0.005089,-0.007437,-0.004712,-0.009241,-0.009473,-0.038477,-630.662720,0.057254,-0.170693,RW
52635,-0.094827,-0.021930,0.008714,-0.070498,-0.136880,-0.038327,-0.061547,0.022022,-0.098437,-0.042384,...,0.010807,0.067275,-0.004712,0.238135,0.131268,0.403188,1361.982910,-0.102622,0.653798,RW
52636,0.084732,0.016217,-0.069920,0.050136,0.152476,0.055310,0.067074,-0.082203,0.089818,-0.000830,...,-0.005089,-0.007437,-0.004712,-0.009241,-0.009473,-0.038477,1411.748047,-0.064780,0.667725,RW


In [45]:
# Separate the rural urban column
string_column = merge4['URBAN_RURAL']

# Concatenate with the demeaned columns
demeaned_df1 = pd.concat([demeaned_df, string_column], axis=1)


In [52]:
demeaned_df1.iloc[:, 3999:4019]

,perc_pixels_in_bin_0,perc_pixels_in_bin_1,perc_pixels_in_bin_2,perc_pixels_in_bin_3,perc_pixels_in_bin_4,perc_pixels_in_bin_5,perc_pixels_in_bin_6,perc_pixels_in_bin_7,perc_pixels_in_bin_8,perc_pixels_in_bin_9,perc_pixels_in_bin_10,perc_pixels_in_bin_11,perc_pixels_in_bin_12,perc_pixels_in_bin_13,perc_pixels_in_bin_14,perc_pixels_in_bin_15,perc_pixels_in_bin_16,perc_pixels_in_bin_17,perc_pixels_in_bin_18,perc_pixels_in_bin_19
0,0.0,-0.130454,-0.108609,-0.101711,-0.083356,-0.042491,-0.033224,-0.035152,-0.028864,-0.024898,-0.029938,0.125248,-0.019963,0.059787,0.206662,0.057360,0.356938,-0.002086,-0.041036,-0.124213
1,0.0,-0.130454,-0.108609,-0.101711,-0.076311,-0.035446,-0.026179,-0.035152,0.024190,-0.024898,-0.029938,0.168547,-0.019963,0.079769,0.173199,0.085735,0.270216,-0.047746,-0.041036,-0.124213
2,0.0,-0.130454,-0.108609,-0.099122,-0.080766,-0.037164,-0.030634,-0.035152,0.058843,0.056429,-0.029938,0.090582,-0.019963,0.051108,0.201144,0.056436,0.270256,-0.047746,-0.041036,-0.124213
3,0.0,-0.130454,-0.108609,-0.101711,-0.083356,-0.039458,-0.027159,-0.035152,0.016815,-0.024898,-0.029938,0.140951,-0.019963,0.064724,0.145179,0.070356,0.322793,0.005129,-0.041036,-0.124213
4,0.0,-0.130454,-0.108609,-0.101711,-0.083356,-0.042491,-0.033224,-0.035152,-0.028864,-0.024898,-0.029938,0.125248,-0.019963,0.059787,0.206662,0.057360,0.356938,-0.002086,-0.041036,-0.124213
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52633,0.0,0.247139,-0.036220,-0.042504,-0.033468,-0.014511,-0.008117,-0.010677,-0.003453,-0.003898,-0.005253,-0.004395,-0.003858,-0.006355,-0.005089,-0.007437,-0.004712,-0.009241,-0.009473,-0.038477
52634,0.0,0.072693,0.011836,0.009679,0.032920,-0.006694,-0.008117,-0.010677,-0.003453,-0.003898,-0.005253,-0.004395,-0.003858,-0.006355,-0.005089,-0.007437,-0.004712,-0.009241,-0.009473,-0.038477
52635,0.0,-0.752861,-0.036220,-0.042504,-0.033468,-0.014511,-0.008117,-0.010677,0.012519,0.012074,0.010470,0.011577,0.012113,-0.006355,0.010807,0.067275,-0.004712,0.238135,0.131268,0.403188
52636,0.0,0.247139,-0.036220,-0.042504,-0.033468,-0.014511,-0.008117,-0.010677,-0.003453,-0.003898,-0.005253,-0.004395,-0.003858,-0.006355,-0.005089,-0.007437,-0.004712,-0.009241,-0.009473,-0.038477


## NL

In [54]:
%%time


from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from joblib import Parallel, delayed
import numpy as np

## only rural
df_rural = demeaned_df1[demeaned_df1['URBAN_RURAL'] == 'R']


# Split the data into training and testing sets
X_train, X_test, y_train, y_r_test = train_test_split(df_rural.iloc[:, 3999:4019], df_rural.iloc[:, [4020, 4021]], test_size=0.2, random_state=42)

# Scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Define a function to train and evaluate a RidgeCV model
def train_evaluate_model(X_train, y_train, X_test, y_r_test, column):
    alphas = [0.001, 0.01, 0.1, 1, 10, 100]  # Custom alpha range
    model = RidgeCV(cv=5, alphas=alphas)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_r_test, y_pred)
    print(f'R-squared score for {column}: {r2}')
    print(f'Selected alpha for {column}: {model.alpha_}')
    return r2, y_pred

# Train and evaluate the models using joblib parallel processing
results = Parallel(n_jobs=-1)(delayed(train_evaluate_model)(X_train, y_train.iloc[:, i], X_test, y_r_test.iloc[:, i], column) for i, column in enumerate(y_train.columns))

# Extract the predicted y values from the results
y_preds_rural = np.concatenate([result[1].reshape(-1, 1) for result in results], axis=1)

##do this for urban

df_urban = demeaned_df1[demeaned_df1['URBAN_RURAL'] == 'U']


# Split the data into training and testing sets
X_train, X_test, y_train, y_u_test = train_test_split(df_urban.iloc[:, 3999:4019], df_urban.iloc[:, [4020, 4021]], test_size=0.2, random_state=42)

# Scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Define a function to train and evaluate a RidgeCV model
def train_evaluate_model(X_train, y_train, X_test, y_u_test, column):
    alphas = [0.001, 0.01, 0.1, 1, 10, 100]  # Custom alpha range
    model = RidgeCV(cv=5, alphas=alphas)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_u_test, y_pred)
    print(f'R-squared score for {column}: {r2}')
    print(f'Selected alpha for {column}: {model.alpha_}')
    return r2, y_pred

# Train and evaluate the models using joblib parallel processing
results = Parallel(n_jobs=-1)(delayed(train_evaluate_model)(X_train, y_train.iloc[:, i], X_test, y_u_test.iloc[:, i], column) for i, column in enumerate(y_train.columns))

# Extract the predicted y values from the results
y_preds_urban = np.concatenate([result[1].reshape(-1, 1) for result in results], axis=1)

y_preds = np.concatenate((y_preds_urban, y_preds_rural), axis=0)
y_true = np.concatenate((y_u_test, y_r_test), axis=0)

print(r2_score(y_true[:,0], y_preds[:,0])) #mpi
print(r2_score(y_true[:,1], y_preds[:,1])) #loginc



0.39221628194089386
0.526068147141447
Wall time: 9.32 s


## mosaiks

In [55]:
%%time


from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from joblib import Parallel, delayed
import numpy as np

## only rural
df_rural = demeaned_df1[demeaned_df1['URBAN_RURAL'] == 'R']


# Split the data into training and testing sets
X_train, X_test, y_train, y_r_test = train_test_split(df_rural.iloc[:, 1:3999], df_rural.iloc[:, [4020, 4021]], test_size=0.2, random_state=42)

# Scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Define a function to train and evaluate a RidgeCV model
def train_evaluate_model(X_train, y_train, X_test, y_r_test, column):
    alphas = [0.001, 0.01, 0.1, 1, 10, 100]  # Custom alpha range
    model = RidgeCV(cv=5, alphas=alphas)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_r_test, y_pred)
    print(f'R-squared score for {column}: {r2}')
    print(f'Selected alpha for {column}: {model.alpha_}')
    return r2, y_pred

# Train and evaluate the models using joblib parallel processing
results = Parallel(n_jobs=-1)(delayed(train_evaluate_model)(X_train, y_train.iloc[:, i], X_test, y_r_test.iloc[:, i], column) for i, column in enumerate(y_train.columns))

# Extract the predicted y values from the results
y_preds_rural = np.concatenate([result[1].reshape(-1, 1) for result in results], axis=1)

##do this for urban

df_urban = demeaned_df1[demeaned_df1['URBAN_RURAL'] == 'U']


# Split the data into training and testing sets
X_train, X_test, y_train, y_u_test = train_test_split(df_urban.iloc[:, 1:3999], df_urban.iloc[:, [4020, 4021]], test_size=0.2, random_state=42)

# Scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Define a function to train and evaluate a RidgeCV model
def train_evaluate_model(X_train, y_train, X_test, y_u_test, column):
    alphas = [0.001, 0.01, 0.1, 1, 10, 100]  # Custom alpha range
    model = RidgeCV(cv=5, alphas=alphas)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_u_test, y_pred)
    print(f'R-squared score for {column}: {r2}')
    print(f'Selected alpha for {column}: {model.alpha_}')
    return r2, y_pred

# Train and evaluate the models using joblib parallel processing
results = Parallel(n_jobs=-1)(delayed(train_evaluate_model)(X_train, y_train.iloc[:, i], X_test, y_u_test.iloc[:, i], column) for i, column in enumerate(y_train.columns))

# Extract the predicted y values from the results
y_preds_urban = np.concatenate([result[1].reshape(-1, 1) for result in results], axis=1)

y_preds = np.concatenate((y_preds_urban, y_preds_rural), axis=0)
y_true = np.concatenate((y_u_test, y_r_test), axis=0)

print(r2_score(y_true[:,0], y_preds[:,0])) # mpi
print(r2_score(y_true[:,1], y_preds[:,1])) #loginc



0.5135535911222013
0.6171150630518721
Wall time: 11min 28s


## mosaiks + nl

In [56]:
%%time


from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from joblib import Parallel, delayed
import numpy as np

## only rural
df_rural = demeaned_df1[demeaned_df1['URBAN_RURAL'] == 'R']


# Split the data into training and testing sets
X_train, X_test, y_train, y_r_test = train_test_split(df_rural.iloc[:, 1:4019], df_rural.iloc[:, [4020, 4021]], test_size=0.2, random_state=42)

# Scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Define a function to train and evaluate a RidgeCV model
def train_evaluate_model(X_train, y_train, X_test, y_r_test, column):
    alphas = [0.001, 0.01, 0.1, 1, 10, 100]  # Custom alpha range
    model = RidgeCV(cv=5, alphas=alphas)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_r_test, y_pred)
    print(f'R-squared score for {column}: {r2}')
    print(f'Selected alpha for {column}: {model.alpha_}')
    return r2, y_pred

# Train and evaluate the models using joblib parallel processing
results = Parallel(n_jobs=-1)(delayed(train_evaluate_model)(X_train, y_train.iloc[:, i], X_test, y_r_test.iloc[:, i], column) for i, column in enumerate(y_train.columns))

# Extract the predicted y values from the results
y_preds_rural = np.concatenate([result[1].reshape(-1, 1) for result in results], axis=1)

##do this for urban

df_urban = demeaned_df1[demeaned_df1['URBAN_RURAL'] == 'U']


# Split the data into training and testing sets
X_train, X_test, y_train, y_u_test = train_test_split(df_urban.iloc[:, 1:4019], df_urban.iloc[:, [4020, 4021]], test_size=0.2, random_state=42)

# Scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Define a function to train and evaluate a RidgeCV model
def train_evaluate_model(X_train, y_train, X_test, y_u_test, column):
    alphas = [0.001, 0.01, 0.1, 1, 10, 100]  # Custom alpha range
    model = RidgeCV(cv=5, alphas=alphas)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_u_test, y_pred)
    print(f'R-squared score for {column}: {r2}')
    print(f'Selected alpha for {column}: {model.alpha_}')
    return r2, y_pred

# Train and evaluate the models using joblib parallel processing
results = Parallel(n_jobs=-1)(delayed(train_evaluate_model)(X_train, y_train.iloc[:, i], X_test, y_u_test.iloc[:, i], column) for i, column in enumerate(y_train.columns))

# Extract the predicted y values from the results
y_preds_urban = np.concatenate([result[1].reshape(-1, 1) for result in results], axis=1)

y_preds = np.concatenate((y_preds_urban, y_preds_rural), axis=0)
y_true = np.concatenate((y_u_test, y_r_test), axis=0)

print(r2_score(y_true[:,0], y_preds[:,0])) # mpi
print(r2_score(y_true[:,1], y_preds[:,1])) # loginc



0.5413415443617267
0.6434074594940717
Wall time: 11min 44s
